# Aprendizado de Máquina — Lista prática E2

## Redução de Dimensionalidade (PCA e t-SNE)

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Duas perguntas nesta lista, e nenhuma delas é "qual método é melhor".

A primeira é sobre **o que cada método preserva** — PCA e t-SNE otimizam coisas
diferentes, e medir as duas coisas mostra que cada um ganha exatamente onde o
outro perde. A segunda revisita a Aula 06:

> **ajustar o PCA no conjunto todo, antes de separar treino e validação, usa
> informação do teste. Isso infla o desempenho medido? Meça antes de responder.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr

import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — PCA pela SVD, à mão

A Lista Teórica E2 mostrou que, se $X = UDV^\top$ é a decomposição em valores
singulares dos dados **centrados**, então as colunas de $V$ são as componentes
principais, os autovalores são $d_j^2/(n-1)$, e as coordenadas são as colunas de
$UD$.

Faça as três coisas e confira contra o `scikit-learn`. O banco é o `digits`:
1 797 imagens $8\times 8$ de dígitos manuscritos, achatadas em 64 colunas.

In [ ]:
dados = load_digits()
X, y = dados.data, dados.target
print(f"digits: {X.shape}, {len(set(y))} classes")

X_c = X - ...                                         # (a) centrar e obrigatorio

U, D, Vt = np.linalg.svd(X_c, full_matrices=False)
autovalores = ...                              # (b) divisor n-1, como o sklearn

pca = PCA().fit(X_c)

print(f"5 primeiros autovalores (SVD)    : {autovalores[:5].round(4)}")
print(f"5 primeiros (sklearn)            : {pca.explained_variance_[:5].round(4)}")
print(f"diferenca maxima nos 60 primeiros: "
      f"{np.abs(autovalores[:60] - pca.explained_variance_[:60]).max():.2e}")

In [ ]:
Z_svd = ...                                           # (a) as coordenadas sao as colunas de UD
Z_skl = pca.transform(X_c)[:, :2]

# os autovetores sao definidos a menos de sinal, entao comparamos os modulos
print(f"diferenca maxima nas coordenadas: "
      f"{np.abs(np.abs(Z_svd) - np.abs(Z_skl)).max():.2e}")

---
## Exercício 2 — quantas componentes bastam

Cada imagem tem 64 pixels, mas os pixels de uma imagem de dígito são fortemente
correlacionados: as bordas são quase sempre pretas, e traços vizinhos aparecem
juntos. Quanta dimensão de verdade existe ali?

In [ ]:
acumulada = ...             # (a)

for alvo in (0.80, 0.90, 0.95, 0.99):
    n_comp = ...               # (b) o primeiro que atinge o alvo
    print(f"{int(alvo * 100)}% da variancia: {n_comp:2d} componentes de {X.shape[1]}")

print(f"\nPCA(n_components=0.90) devolve: {PCA(n_components=...).fit(X_c).n_components_}")   # (c)

---
## Exercício 3 — o que cada método preserva

Agora compare PCA e t-SNE por **duas** medidas diferentes:

- **estrutura global**: a correlação de Spearman entre as distâncias no espaço
  original e as distâncias no mapa, sobre todos os pares;
- **estrutura local**: a fração dos 10 vizinhos mais próximos de cada ponto no
  espaço original que continuam entre os 10 mais próximos no mapa.

Use uma subamostra de 800 pontos — o t-SNE tem custo quadrático.

In [ ]:
sub = np.random.default_rng(2026).choice(len(X), 800, replace=False)
X_s, y_s = X[sub], y[sub]

Z_pca = PCA(n_components=2).fit_transform(X_s)
Z_tsne = TSNE(n_components=2, perplexity=..., random_state=2026,
              init="pca").fit_transform(X_s)                       # (a)

dist_alto = pdist(X_s)


def preservacao_vizinhanca(X_alto, Z_baixo, k=10):
    viz_alto = NearestNeighbors(n_neighbors=k + 1).fit(X_alto) \
        .kneighbors(X_alto, return_distance=False)[:, 1:]          # [:, 1:] descarta o proprio ponto
    viz_baixo = NearestNeighbors(n_neighbors=k + 1).fit(Z_baixo) \
        .kneighbors(Z_baixo, return_distance=False)[:, 1:]
    return np.mean([len(...) / k                     # (a) vizinhos em comum
                    for a, b in zip(viz_alto, viz_baixo)])


print("          correlacao global    preservacao local")
for nome, Z in (("PCA", Z_pca), ("t-SNE", Z_tsne)):
    global_ = ...         # (b)
    local = preservacao_vizinhanca(X_s, Z)
    print(f"  {nome:6s}      {global_:.4f}              {local:.4f}")

> **Sua vez.** Desenhe os dois mapas lado a lado, coloridos pelo dígito
> verdadeiro (`c=y_s`). Em qual dos dois os dez algarismos aparecem separados?

---
## Exercício 4 — o PCA fora da dobra é vazamento grave?

Ajustar o PCA no conjunto completo, antes de separar treino e validação, **usa
informação do teste**: as direções principais dependem de todos os pontos. Pela
intuição, isso parece grave — o PCA aprende de todas as colunas ao mesmo tempo.

Mas o critério da Aula 06 não é "quanto a etapa aprende", é **"a etapa olha o
$Y$?"**. O PCA não olha. Vamos medir o efeito, no cenário mais cruel possível:
$y$ de ruído puro, em que o $R^2$ verdadeiro é zero e qualquer inflação fica
visível.

In [ ]:
cv = skm.KFold(5, shuffle=True, random_state=0)

print("       cenario           PCA fora    PCA dentro   diferenca")
for n, d, k in [(60, 200, 10), (60, 1000, 20), (100, 500, 15), (200, 2000, 30)]:
    rng = np.random.default_rng(2026)
    fora, dentro = [], []

    for _ in range(20):
        X_r = rng.normal(size=(n, d))
        y_r = ...                                 # (a) NENHUMA relacao com X

        # ERRADO: o PCA ve o conjunto todo, depois valida
        # svd_solver="full" torna o resultado deterministico: com d grande, o
        # padrao "auto" escolhe o solucionador randomizado, que varia entre execucoes
        X_p = PCA(n_components=k, svd_solver="full").fit_transform(...)   # (b)
        fora.append(skm.cross_val_score(skl.LinearRegression(), X_p, y_r,
                                        cv=cv, scoring="r2").mean())

        # CERTO: o PCA e um passo do pipeline, refeito em cada dobra
        tubo = Pipeline([("pca", ...),   # (c)
                         ("mqo", skl.LinearRegression())])
        dentro.append(skm.cross_val_score(tubo, X_r, y_r, cv=cv, scoring="r2").mean())

    print(f"  n={n:3d} d={d:4d} k={k:2d}    {np.mean(fora):+.4f}     "
          f"{np.mean(dentro):+.4f}     {...:+.4f}")   # (d)

> **Sua vez.** Repita o primeiro cenário trocando `y_r` por um $y$ que **dependa**
> de `X_r` — por exemplo, `X_r @ beta + ruido` com `beta` esparso. A diferença
> entre fazer o PCA fora e dentro continua negativa?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | a PCA pela SVD à mão bate o `scikit-learn` até $10^{-13}$, em autovalores e coordenadas |
| 2 | 21 das 64 colunas do `digits` carregam 90% da variância |
| 3 | o PCA preserva melhor a estrutura **global** (0,5838 contra 0,4634) |
| 3 | o t-SNE preserva **três vezes mais** vizinhança local (0,6466 contra 0,1954) |
| 4 | o PCA fora da dobra **piora** o $R^2$ em 0,28 a 1,02 — nunca infla |

**A seguir.** A Aula E3 fecha o curso aplicando quase tudo a um problema real de
texto: representar mensagens como vetores, classificar, e escolher a métrica
certa para um problema desbalanceado.